# Revise one existing FDA indication

Cotellic historical replay: detect a change to one existing melanoma indication, isolate target-specific evidence, and propose a reviewable patch without writing to `moalmanac-db`.

In [1]:
import json, os
from pathlib import Path
from pprint import pprint
from dotenv import find_dotenv, load_dotenv
from moalmanac_fda_curation.core.revise_indication import (
    assess_update, build_assessment_prompt, build_proposal_prompt,
    events_since, propose_revision, refresh_changelog,
)

## Load the Anthropic API key

This reports availability without displaying the secret.

In [2]:
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("No .env file found")
load_dotenv(env_path)
if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(f"ANTHROPIC_API_KEY is missing from {env_path}")
print(f"Loaded ANTHROPIC_API_KEY from {env_path}")

Loaded ANTHROPIC_API_KEY from /Users/sabrina/Documents/VA/MOAlmanac/moalmanac-fda-curation/.env


## Cotellic historical-replay inputs

The 2022 event adds `adult` to the melanoma indication and separately introduces histiocytic neoplasms. The latter must not enter this indication's patch.

In [3]:
PROJECT_ROOT = Path(env_path).parent
WORK_DIR = PROJECT_ROOT / "analyses/revisions/cotellic-historical-replay"
DOCUMENT_JSON = WORK_DIR / "document.json"
LAST_CURATION_DATE = "2018-01-26"
indication = {
    "id": "ind:fda.cotellic:0",
    "document_id": "doc:fda.cotellic",
    "indication": "COTELLIC is a kinase inhibitor indicated for the treatment of patients with unresectable or metastatic melanoma with a BRAF V600E or V600K mutation, in combination with vemurafenib.",
    "initial_approval_date": "2015-11-10",
    "initial_approval_url": "https://www.accessdata.fda.gov/drugsatfda_docs/label/2015/206192s000lbl.pdf",
    "description": "The U.S. Food and Drug Administration granted approval to cobimetinib in combination with vemurafenib for the treatment of patients with unresectable or metastatic melanoma with a BRAF V600E or V600K mutation.",
    "raw_biomarkers": "BRAF V600E or V600K",
    "raw_cancer_type": "unresectable or metastatic melanoma",
    "raw_therapeutics": "Cotellic (cobimetinib) in combination with vemurafenib",
}
document = {"id": "doc:fda.cotellic", "drug_name_brand": "Cotellic", "drug_name_generic": "cobimetinib", "identification_number": 206192, "urls": [indication["initial_approval_url"]]}
WORK_DIR.mkdir(parents=True, exist_ok=True)
DOCUMENT_JSON.write_text(json.dumps(document, indent=2) + "\n")
pprint(indication)

{'description': 'The U.S. Food and Drug Administration granted approval to '
                'cobimetinib in combination with vemurafenib for the treatment '
                'of patients with unresectable or metastatic melanoma with a '
                'BRAF V600E or V600K mutation.',
 'document_id': 'doc:fda.cotellic',
 'id': 'ind:fda.cotellic:0',
 'indication': 'COTELLIC is a kinase inhibitor indicated for the treatment of '
               'patients with unresectable or metastatic melanoma with a BRAF '
               'V600E or V600K mutation, in combination with vemurafenib.',
 'initial_approval_date': '2015-11-10',
 'initial_approval_url': 'https://www.accessdata.fda.gov/drugsatfda_docs/label/2015/206192s000lbl.pdf',
 'raw_biomarkers': 'BRAF V600E or V600K',
 'raw_cancer_type': 'unresectable or metastatic melanoma',
 'raw_therapeutics': 'Cotellic (cobimetinib) in combination with vemurafenib'}


## 1. Refresh and inspect post-curation events

In [4]:
refreshed = refresh_changelog(DOCUMENT_JSON, WORK_DIR)
candidate_events = events_since(refreshed["changelog"], LAST_CURATION_DATE)
print(refreshed["markdown_path"])
pprint(candidate_events)

downloading 20151110 http://www.accessdata.fda.gov/drugsatfda_docs/label/2015/206192s000lbl.pdf
wrote /Users/sabrina/Documents/VA/MOAlmanac/moalmanac-fda-curation/analyses/revisions/cotellic-historical-replay/historical-labels/Cotellic-nda206192/2015-11-10-ORIG-1-206192s000lbl.pdf
wrote /Users/sabrina/Documents/VA/MOAlmanac/moalmanac-fda-curation/analyses/revisions/cotellic-historical-replay/historical-labels/Cotellic-nda206192/2015-11-10-ORIG-1-206192s000lbl.md
downloading 20180126 http://www.accessdata.fda.gov/drugsatfda_docs/label/2018/206192s002lbl.pdf
wrote /Users/sabrina/Documents/VA/MOAlmanac/moalmanac-fda-curation/analyses/revisions/cotellic-historical-replay/historical-labels/Cotellic-nda206192/2018-01-26-SUPPL-2-206192s002lbl.pdf
wrote /Users/sabrina/Documents/VA/MOAlmanac/moalmanac-fda-curation/analyses/revisions/cotellic-historical-replay/historical-labels/Cotellic-nda206192/2018-01-26-SUPPL-2-206192s002lbl.md
downloading 20220728 http://www.accessdata.fda.gov/drugsatfda_do

## 2. Assess and isolate target-specific spans

The LLM selects an event and exact target-specific before/after quotes. Python verifies both quotes verbatim against the canonical event. `relevant_events` retains complete audit evidence; `scoped_evidence` is the only evidence sent to the proposal step. The next cell prints the exact assessment prompt before calling the model.

In [5]:
assessment_prompt = build_assessment_prompt(indication, candidate_events)
print(assessment_prompt)
assessment = assess_update(indication, candidate_events)
pprint(assessment["assessment"])
print("\nVerified scoped evidence:")
pprint(assessment["scoped_evidence"])
print("\nVerification errors:", assessment["verification_errors"])

{'reasoning': 'Event 3 modifies the existing melanoma indication by adding the '
              "qualifier 'adult' to the patient population. The indication "
              "previously applied to 'patients' generally but now explicitly "
              "specifies 'adult patients'. This is a clinically meaningful "
              'change as it restricts the approved population by age. The '
              'event also adds a new separate indication for histiocytic '
              'neoplasms, but that does not affect the existing melanoma '
              'indication being reviewed.',
 'relevant_event_numbers': [3],
 'scoped_evidence': [{'event_number': 3,
                      'target_after_quote': 'COTELLIC® is indicated for the '
                                            'treatment of adult patients with '
                                            'unresectable or metastatic '
                                            'melanoma with a BRAF V600E or '
                                  

### Expected assessment

Expect `updated`, event 3, and scoped quotes containing only the melanoma sentence. The change is the adult qualifier. The histiocytic-neoplasms sentence should be absent from `scoped_evidence`.

## 3. Propose from scoped evidence only

Expected LLM replacements add `adult` only to `indication` and `description`. Separately, Python deterministically updates `initial_approval_date` and `initial_approval_url` from the latest verified target event. For this replay they should become `2022-10-28` and event 3's label URL. The proposal model does not receive the complete event, preventing the separate histiocytic indication from leaking into this record. The next cell prints the exact proposal prompt before calling the model.

In [6]:
if assessment["assessment"]["status"] == "updated" and assessment["verified"]:
    proposal_prompt = build_proposal_prompt(indication, assessment["scoped_evidence"])
    print(proposal_prompt)
    proposal = propose_revision(indication, assessment)
    pprint(proposal)
else:
    print("No verified update to propose.")

{'patch': {'description': 'The U.S. Food and Drug Administration granted '
                          'approval to cobimetinib in combination with '
                          'vemurafenib for the treatment of adult patients '
                          'with unresectable or metastatic melanoma with a '
                          'BRAF V600E or V600K mutation.',
           'indication': 'COTELLIC is a kinase inhibitor indicated for the '
                         'treatment of adult patients with unresectable or '
                         'metastatic melanoma with a BRAF V600E or V600K '
                         'mutation, in combination with vemurafenib.'},
 'proposal': {'summary': "Updated indication and description to specify 'adult "
                         "patients' instead of just 'patients' based on label "
                         'revision event 3.',
              'uncertainties': [],
              'updates': [{'field': 'indication',
                           'new_value': 'COTEL

## Optional: save after curator inspection

In [ ]:
# output = WORK_DIR / "revision-proposal.json"
# output.write_text(json.dumps(proposal, indent=2) + "\n")
# output